# Multi-Agent Real Estate Analysis with CrewAI

This notebook is the final stage of the real estate AI project. It uses CrewAI and Google's Gemini model to coordinate two specialized agents: a Real Estate Data Analyst and a Residential Property Price Analyst. The workflow cleans and samples active property listings, exposes the sample through a custom CrewAI tool, and runs a hierarchical multi-agent process to produce a structured market and pricing analysis.

> **Security note:** The Gemini API key is loaded from the `GEMINI_API_KEY` environment variable and is never stored in this notebook.


In [ ]:
# Install the packages needed for CrewAI and its tools
!pip install -q -U crewai
!pip install -q -U duckduckgo-search
!pip install -q -U "crewai[tools]"

# Import pandas for cleaning and sampling the dataset
import pandas as pd

# Import the CrewAI classes needed to create agents, tasks, and the crew
from crewai import Agent, Task, Crew, Process, LLM

# Import BaseTool so we can create a custom CSV reader tool
from crewai.tools import BaseTool

This cell is responsible for setting up the necessary software environment. It uses `!pip install -q -U` commands to install three key Python packages:

*   `crewai`: This is the core library for building and orchestrating AI agents in a multi-agent system. It allows defining agents, tasks, and crews.
*   `duckduckgo-search`: This package provides a tool for agents to perform web searches, enabling them to gather external information if needed. While not explicitly used in the custom tool logic here, it's a common dependency for agents that require external knowledge.
*   `crewai[tools]`: This installs additional tools and utilities specifically designed to integrate with CrewAI agents, expanding their capabilities.

Following the installations, it imports several modules:

*   `pandas as pd`: Imports the powerful `pandas` library, aliased as `pd`, which is crucial for efficient data manipulation, cleaning, and analysis of the real estate dataset.
*   `Agent`, `Task`, `Crew`, `Process`, `LLM` from `crewai`: These are fundamental classes for defining the agents themselves, the specific jobs they perform (`Task`), how they work together (`Crew`), the execution flow (`Process`), and the underlying large language model (`LLM`) that powers their intelligence.
*   `BaseTool` from `crewai.tools`: This class serves as the foundation for creating custom tools that agents can use to interact with specific data sources or perform custom actions, as demonstrated later with `CsvStringReaderTool`.

In [ ]:
import os

# Load the Gemini API key from an environment variable
gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError("Set the GEMINI_API_KEY environment variable before running this notebook.")

# Configure the Gemini model used by the CrewAI agents
gemini_llm = LLM(
    model="gemini/gemini-3.6-flash",
    api_key=gemini_api_key,
    temperature=0.0
)

print("Gemini LLM configured successfully.")


### Configure the language model

The Gemini model is configured through CrewAI's `LLM` interface. The API key is read from the `GEMINI_API_KEY` environment variable so credentials are not committed to the repository. A temperature of `0.0` is used to keep the analytical workflow more consistent and reproducible.


In [ ]:
# Path to the original real estate dataset in the project data folder
file_path = "../data/realtor-data.zip.csv"

# Path where the cleaned dataset will be saved
cleaned_file_path = "../data/realtor_houses_for_sale_cleaned.csv"

try:
    # Load the original CSV file into a pandas DataFrame
    real_estate_df = pd.read_csv(
        file_path,
        low_memory=False
    )

    print("Original dataset loaded successfully.")
    print(f"Original rows: {len(real_estate_df):,}")

    # Create a copy so the original DataFrame is not changed
    houses_for_sale = real_estate_df.copy()

    # Standardize the status column
    houses_for_sale["status"] = (
        houses_for_sale["status"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Keep only listings that are currently marked for sale
    houses_for_sale = houses_for_sale[
        houses_for_sale["status"] == "for_sale"
    ].copy()

    # Convert important property fields into numeric values
    numeric_columns = [
        "price",
        "bed",
        "bath",
        "acre_lot",
        "house_size"
    ]

    for column in numeric_columns:
        houses_for_sale[column] = pd.to_numeric(
            houses_for_sale[column],
            errors="coerce"
        )

    # Remove listings that are missing information required for analysis
    houses_for_sale = houses_for_sale.dropna(
        subset=[
            "price",
            "bed",
            "bath",
            "house_size",
            "city",
            "state"
        ]
    )

    # Remove listings with impossible or unusable property values
    houses_for_sale = houses_for_sale[
        (houses_for_sale["price"] > 0)
        & (houses_for_sale["bed"] >= 1)
        & (houses_for_sale["bath"] >= 1)
        & (houses_for_sale["house_size"] > 0)
    ].copy()

    # Calculate the listing price per square foot
    houses_for_sale["price_per_sqft"] = (
        houses_for_sale["price"]
        / houses_for_sale["house_size"]
    )

    # Remove likely data errors with extremely low or high values
    houses_for_sale = houses_for_sale[
        houses_for_sale["price_per_sqft"].between(
            10,
            5000
        )
    ].copy()

    # Remove exact duplicate records
    houses_for_sale = houses_for_sale.drop_duplicates()

    # Save the cleaned version for documentation and later use
    houses_for_sale.to_csv(
        cleaned_file_path,
        index=False
    )

    print("Dataset cleaning completed successfully.")
    print(f"Cleaned rows: {len(houses_for_sale):,}")
    print(f"Cleaned file saved to: {cleaned_file_path}")

    display(houses_for_sale.head())

except FileNotFoundError:
    print(
        f"File not found at {file_path}. "
        "Check that the dataset exists in the project data folder."
    )

except Exception as error:
    print(f"An error occurred while cleaning the dataset: {error}")

This cell is dedicated to robust data cleaning and preprocessing of the real estate dataset, which is crucial for ensuring the quality and reliability of the agent's analysis.

1.  **Load Data**: It starts by defining `file_path` and `cleaned_file_path`, then attempts to load the large `realtor-data.zip.csv` file into a pandas DataFrame `real_estate_df` using `pd.read_csv()`. `low_memory=False` is used to prevent potential DtypeWarning when processing large files.
2.  **Filter 'for_sale' Listings**: A copy of the DataFrame `houses_for_sale` is created. The `status` column is standardized to lowercase and stripped of whitespace, and then filtered to include only listings with `status == "for_sale"`. This ensures the analysis focuses solely on active listings.
3.  **Convert to Numeric**: Key columns like `price`, `bed`, `bath`, `acre_lot`, and `house_size` are converted to numeric types using `pd.to_numeric(errors="coerce")`. `errors="coerce"` will turn any non-convertible values into `NaN` (Not a Number), which can then be handled.
4.  **Handle Missing Values**: `houses_for_sale.dropna(subset=...)` removes rows where essential columns (`price`, `bed`, `bath`, `house_size`, `city`, `state`) have `NaN` values. This ensures that only complete records relevant for core analysis are retained.
5.  **Remove Impossible Values**: The code filters out listings with unrealistic or erroneous values, such as prices less than or equal to zero, bedrooms or bathrooms less than one, or house sizes less than or equal to zero. This step helps in removing data entry errors or irrelevant entries.
6.  **Calculate `price_per_sqft`**: A new feature, `price_per_sqft`, is calculated by dividing `price` by `house_size`. This metric is often a better indicator of value than raw price alone.
7.  **Filter Outliers for `price_per_sqft`**: Listings with `price_per_sqft` outside a reasonable range (10 to 5000) are removed. This helps to eliminate extreme outliers that could skew the analysis, potentially caused by data errors or highly unusual properties.
8.  **Remove Duplicates**: `houses_for_sale.drop_duplicates()` ensures that each listing record is unique, preventing overcounting or biased statistics from duplicate entries.
9.  **Save Cleaned Data**: The processed DataFrame is saved to a new CSV file (`realtor_houses_for_sale_cleaned.csv`) without the DataFrame index. This provides a clean, persistent dataset for future use.
10. **Display Head**: `display(houses_for_sale.head())` shows the first few rows of the cleaned data, allowing for a quick visual inspection of the results.
11. **Error Handling**: A `try-except` block is used to gracefully handle `FileNotFoundError` if the initial CSV is not found and other `Exception` types during the cleaning process, providing informative messages to the user.

In [ ]:
# Keep only the fields that are useful for the agent analysis
analysis_columns = [
    "status",
    "price",
    "bed",
    "bath",
    "acre_lot",
    "city",
    "state",
    "zip_code",
    "house_size",
    "price_per_sqft"
]

# Create a reduced DataFrame for the CrewAI agents
houses_for_agents = houses_for_sale[
    analysis_columns
].copy()

# Use 100 listings to reduce API token usage
sample_size = min(
    100,
    len(houses_for_agents)
)

# Create a reproducible random sample
# The same random_state produces the same sample each time
houses_sample = houses_for_agents.sample(
    n=sample_size,
    random_state=42
)

# Path where the sampled CSV will be saved
sample_file_path = "../data/realtor_houses_for_sale_sample.csv"

# Save the sample as a CSV file
houses_sample.to_csv(
    sample_file_path,
    index=False
)

# Read the sampled CSV file as one long text string
with open(
    sample_file_path,
    "r",
    encoding="utf-8"
) as file:
    csv_string = file.read()

print("Real estate sample prepared successfully.")
print(f"Rows provided to CrewAI: {len(houses_sample):,}")
print(f"Characters in CSV string: {len(csv_string):,}")
print("\nFirst 500 characters:")
print(csv_string[:500])

This cell is crucial for preparing a manageable and focused dataset for the CrewAI agents, optimizing for performance and API token usage:

1.  **Select Relevant Columns**: `analysis_columns` is defined to include only the fields that are directly pertinent to the real estate analysis tasks (e.g., `price`, `bed`, `bath`, `city`, `state`, `price_per_sqft`). This reduces the data volume and helps the agents focus on key information.
2.  **Create Subset DataFrame**: A new DataFrame `houses_for_agents` is created, containing only these selected columns from the `houses_for_sale` DataFrame.
3.  **Sampling**: To manage the computational load and API token costs associated with processing large datasets by LLMs, a smaller `sample_size` of 100 listings is chosen. `houses_for_agents.sample(n=sample_size, random_state=42)` extracts a random subset of this size. `random_state=42` ensures that the same random sample is selected every time the code is run, making the results reproducible.
4.  **Save Sampled Data**: The sampled data is saved to a new CSV file (`realtor_houses_for_sale_sample.csv`). This intermediate step provides a persistent, smaller dataset that can be easily accessed.
5.  **Read CSV as String**: The content of the sampled CSV file is then read into a single, long string variable called `csv_string`. This is a critical step because the custom tool (`CsvStringReaderTool`) that will be created next expects the data as a single string, which is an efficient way to pass a static dataset to an agent's tool without requiring file system access or database queries during the agent's execution.

In [ ]:
# Create a custom CrewAI tool that gives an agent access
# to the sampled real estate data as a text string
class CsvStringReaderTool(BaseTool):
    name: str = "Real Estate CSV String Reader"

    description: str = (
        "Returns the complete cleaned sample of 100 for-sale residential "
        "real estate listings as one CSV-formatted text string. Call this "
        "tool only once. It does not execute SQL queries."
    )

    # Store the CSV text inside the tool
    csv_data: str

    def _run(self, query: str = None) -> str:
        """
        Return the complete sampled real estate CSV string.

        The query parameter is accepted because CrewAI tools expect an
        input, but this tool always returns the same prepared CSV string.
        """
        return self.csv_data


# Create the tool and pass the sampled CSV text into it
csv_reader_tool = CsvStringReaderTool(
    csv_data=csv_string
)

print("CSV reader tool created successfully.")
print(
    f"Characters stored in tool: "
    f"{len(csv_reader_tool.csv_data):,}"
)

This cell defines a custom tool, `CsvStringReaderTool`, which is a key component for allowing CrewAI agents to access the prepared real estate data.

1.  **Custom Tool Definition**: `class CsvStringReaderTool(BaseTool):` defines a new tool by inheriting from `BaseTool`. This is the standard way to create custom functionalities for CrewAI agents.
2.  **Tool Metadata**:
    *   `name: str = "Real Estate CSV String Reader"`: Provides a clear, descriptive name for the tool, which agents will use to refer to it.
    *   `description: str = "Returns the complete cleaned sample..."`: Offers a detailed explanation of what the tool does, its purpose, and instructions for how agents should use it (e.g., call only once, does not execute SQL). This description is crucial for the LLM to understand when and how to invoke the tool.
3.  **Data Storage (`csv_data`)**: `csv_data: str` declares an attribute to store the entire sampled CSV content as a string directly within the tool instance. This makes the data immediately available to the tool's `_run` method.
4.  **`_run` Method**: This method contains the core logic of the tool.
    *   `def _run(self, query: str = None) -> str:`: This method is called when an agent uses the tool. It accepts a `query` parameter (a standard requirement for CrewAI tools, even if not explicitly used here) and is designed to return a string.
    *   `return self.csv_data`: The method simply returns the `csv_string` that was loaded into the tool. This means the agent gets the entire 100-record CSV sample directly as text.
5.  **Tool Instantiation**: `csv_reader_tool = CsvStringReaderTool(csv_data=csv_string)` creates an actual instance of the tool, passing the `csv_string` (generated in the previous cell) to its `csv_data` attribute. This makes the data ready for the tool to serve to the agents.

**Purpose**: This custom tool acts as a controlled interface. Instead of allowing agents to perform arbitrary file reads or database queries, it provides them with a pre-defined, static dataset in a specific format (CSV string). This ensures that the agents work with consistent data, prevents unintended external access, and helps manage token usage by providing a bounded input.

In [ ]:
# ---------------------------------------------------------
# AGENT 1: REAL ESTATE DATA ANALYST
# ---------------------------------------------------------

real_estate_analyst = Agent(
    role="Real Estate Data Analyst",

    goal=(
        "Analyze the cleaned sample of for-sale residential real estate "
        "listings and identify pricing patterns, geographic differences, "
        "property characteristics, missing values, unusual records, and "
        "relationships between listing price and property features."
    ),

    backstory=(
        "You are an experienced residential real estate data analyst. "
        "You examine housing datasets and explain how location, house "
        "size, bedrooms, bathrooms, acreage, and price per square foot "
        "relate to listing prices. You separate findings supported by "
        "the dataset from assumptions and clearly explain limitations."
    ),

    verbose=True,
    allow_delegation=False,
    llm=gemini_llm,
    tools=[csv_reader_tool]
)


# ---------------------------------------------------------
# TASK 1: ANALYZE THE REAL ESTATE SAMPLE
# ---------------------------------------------------------

analyze_real_estate_task = Task(
    description="""
Use the Real Estate CSV String Reader tool to examine the cleaned real
estate sample.

Call the CSV reader tool only one time. The tool returns the complete
100-record sample. Do not call the tool repeatedly and do not attempt
to execute SQL because the tool returns a static CSV-formatted string.

The dataset contains residential listings that were filtered to include
only records with a status of "for_sale".

The available fields are:

- status
- price
- bed
- bath
- acre_lot
- city
- state
- zip_code
- house_size
- price_per_sqft

Complete the following analysis:

1. DATASET OVERVIEW
- Explain what the dataset contains.
- State the number of records and columns.
- Explain that the data is a 100-record sample.
- Explain the limitations of using a sample.

2. DATA QUALITY
- Identify missing or blank values.
- Identify suspicious or unusual values.
- Discuss possible outliers.
- Do not invent values that cannot be supported by the CSV.

3. SUMMARY STATISTICS
Analyze the following numeric fields:

- price
- bed
- bath
- acre_lot
- house_size
- price_per_sqft

Report, where possible:

- count
- mean
- median
- minimum
- maximum
- whether the distribution appears skewed

4. PRICE RELATIONSHIPS
Analyze how listing price appears to relate to:

- house size
- bedroom count
- bathroom count
- acreage
- price per square foot
- city
- state
- ZIP code

5. LOCATION ANALYSIS
- Compare listing prices across the locations in the sample.
- Identify higher-priced and lower-priced locations.
- Compare median price and median price per square foot where possible.
- Explain that locations with small sample counts should be interpreted
  cautiously.

6. PROPERTY CHARACTERISTICS
- Compare smaller and larger homes.
- Compare different bedroom and bathroom counts.
- Discuss whether greater acreage consistently corresponds to higher price.
- Identify characteristics associated with higher listing prices.

7. OUTLIERS AND MARKET SEGMENTS
- Identify unusually expensive and inexpensive listings.
- Identify unusually large or small homes.
- Discuss possible entry-level, mid-priced, and luxury listings.
- Distinguish possible data errors from real market variation.

8. FINAL FINDINGS
Provide:

- clear headings
- concise tables where useful
- bullet-pointed findings
- a conclusion
- a limitations section

Only use the supplied CSV sample. Do not use external real estate information.
Do not invent unsupported statistics or property details.
""",

    expected_output=(
        "A structured real estate data analysis report containing a dataset "
        "overview, data-quality review, summary statistics, geographic "
        "comparisons, price relationships, outliers, key findings, and "
        "limitations."
    ),

    agent=real_estate_analyst
)

print("Real Estate Data Analyst and Task 1 created.")

This cell defines the first agent, `real_estate_analyst`, and its corresponding task, `analyze_real_estate_task`. This agent is the initial point of contact with the raw (but cleaned) data and is responsible for foundational data analysis.

### `real_estate_analyst` Agent Definition:

*   **`role="Real Estate Data Analyst"`**: Assigns a professional identity to the agent, guiding its persona and expertise.
*   **`goal=(...)`**: Defines the overarching objective of this agent: to thoroughly analyze the real estate sample, identify patterns, relationships, and data quality issues. This helps the LLM understand the agent's ultimate purpose.
*   **`backstory=(...)`**: Provides context about the agent's experience and approach, further refining its persona. It emphasizes expertise in examining housing datasets and explaining factors influencing pricing, while distinguishing findings from assumptions.
*   **`verbose=True`**: Enables detailed logging of the agent's thought process, allowing you to see how it plans and executes its task.
*   **`allow_delegation=False`**: This agent is not allowed to delegate parts of its task to other agents, ensuring it completes its assigned work independently.
*   **`llm=gemini_llm`**: Assigns the previously configured `gemini_llm` as the underlying large language model that will power this agent's intelligence and reasoning.
*   **`tools=[csv_reader_tool]`**: Equips the agent with the `csv_reader_tool`, allowing it to access the sampled real estate data. Without this tool, the agent would not be able to 'see' or process the data.

### `analyze_real_estate_task` Task Definition:

*   **`description=(...)`**: This is a highly detailed set of instructions for the agent, outlining specific steps for the real estate analysis. It instructs the agent to:
    *   Use the `Real Estate CSV String Reader` tool *only once* to get the data.
    *   Perform a **Dataset Overview** (explaining content, record count, sample limitations).
    *   Conduct **Data Quality** assessment (missing values, suspicious entries, outliers).
    *   Generate **Summary Statistics** for numeric fields (count, mean, median, min, max, skew).
    *   Analyze **Price Relationships** with house size, bedrooms, bathrooms, etc.
    *   Perform **Location Analysis** (comparing prices across cities/states).
    *   Examine **Property Characteristics** (small vs. large homes, acreage impact).
    *   Identify **Outliers and Market Segments** (expensive/inexpensive listings, luxury segments).
    *   Produce **Final Findings** with clear headings, tables, bullet points, conclusion, and limitations.
    *   Crucially, it explicitly states **not to use external real estate information** or invent unsupported statistics, ensuring the analysis is purely data-driven from the provided sample.
*   **`expected_output=(...)`**: Specifies the desired format and content of the final report from this task. This helps guide the agent towards generating a structured and comprehensive output.
*   **`agent=real_estate_analyst`**: Assigns this task to the `real_estate_analyst` agent, indicating that this agent is responsible for completing this specific piece of work.

In [ ]:
# ---------------------------------------------------------
# AGENT 2: RESIDENTIAL PROPERTY PRICE ANALYST
# ---------------------------------------------------------

property_price_analyst = Agent(
    role="Residential Property Price Analyst",

    goal=(
        "Use the Real Estate Data Analyst's completed report to evaluate "
        "listing-price patterns, create reasonable comparative price ranges, "
        "and explain the property characteristics that appear to affect value."
    ),

    backstory=(
        "You are a residential property pricing analyst who interprets "
        "comparable listing information. You consider location, square "
        "footage, bedrooms, bathrooms, acreage, and price per square foot. "
        "You understand that this analysis is exploratory and is not a "
        "licensed appraisal or a validated machine-learning prediction."
    ),

    verbose=True,
    allow_delegation=False,
    llm=gemini_llm
)


# ---------------------------------------------------------
# TASK 2: EVALUATE PROPERTY PRICES
# ---------------------------------------------------------

estimate_property_prices_task = Task(
    description="""
Use the completed Real Estate Data Analyst report as context to evaluate
residential listing prices.

Do not call the CSV tool. Base this task on the first agent's completed
analysis and the property examples included in that report.

Complete the following:

1. PRICE-ESTIMATION APPROACH
Explain how comparative property prices may be evaluated using:

- state
- city
- ZIP code
- house size
- bedrooms
- bathrooms
- acreage
- price per square foot

Explain that this is an AI-generated comparative pricing exercise. It is
not a trained machine-learning model, licensed appraisal, or guarantee of
market value.

2. MARKET SEGMENTATION
Use the distribution described by the first agent to divide the sample into
reasonable market groups such as:

- lower-priced or entry-level listings
- mid-priced listings
- higher-priced listings
- luxury listings

Base the categories on the sample rather than fixed national price ranges.

3. COMPARABLE-LISTING METHOD
Explain how comparable properties can be evaluated using:

- similar location
- similar house size
- similar bedroom and bathroom counts
- similar acreage
- similar price per square foot

Identify which characteristics appear most useful in this sample.

4. REPRESENTATIVE PROPERTY EVALUATIONS
Using only properties specifically identified in the first agent's report,
select a small number of representative examples.

For each example, include where supported:

- city and state
- bedrooms
- bathrooms
- house size
- acreage
- listed price
- estimated reasonable price range
- whether the listing appears below, near, or above the estimated range
- explanation of the factors influencing the evaluation

Do not invent properties or property details that were not included in the
first agent's report.

5. HIGH- AND LOW-PRICE OBSERVATIONS
Discuss:

- unusually expensive listings relative to house size
- unusually inexpensive listings relative to house size
- unusually high or low price per square foot
- possible data anomalies

6. LIMITATIONS
Explain that:

- the analysis uses a 100-record sample
- the dataset does not contain every property feature
- property age and condition are unavailable
- school district information is unavailable
- renovations and amenities are unavailable
- neighborhood quality is unavailable
- taxes and local market demand are unavailable
- this is not financial advice or a professional appraisal

7. FINAL OUTPUT
Provide:

- executive summary
- methodology
- market segments
- representative property evaluations
- important pricing factors
- limitations
- final conclusion

Only use the first agent's completed report. Do not use outside information
and do not invent unsupported property characteristics.
""",

    expected_output=(
        "A structured residential property pricing report containing an "
        "executive summary, comparative pricing methodology, market segments, "
        "representative property evaluations, pricing observations, major "
        "value factors, limitations, and a conclusion."
    ),

    agent=property_price_analyst,

    # Give Task 2 access to the completed output from Task 1
    context=[analyze_real_estate_task]
)

print("Residential Property Price Analyst and Task 2 created.")

This cell defines the second agent, `property_price_analyst`, and its corresponding task, `estimate_property_prices_task`. This agent builds upon the analysis performed by the first agent, focusing on property valuation and market segmentation.

### `property_price_analyst` Agent Definition:

*   **`role="Residential Property Price Analyst"`**: Assigns a specialized identity focused on property pricing.
*   **`goal=(...)`**: Sets the agent's objective: to evaluate pricing patterns, create comparative price ranges, and explain factors affecting value, all based on the *first agent's report*.
*   **`backstory=(...)`**: Provides context, emphasizing expertise in interpreting comparable listing information and understanding the exploratory nature of the analysis (not a licensed appraisal).
*   **`verbose=True`**: Enables detailed logging of the agent's internal thought processes.
*   **`allow_delegation=False`**: Prevents this agent from delegating its tasks.
*   **`llm=gemini_llm`**: Assigns the `gemini_llm` as its underlying intelligence.

### `estimate_property_prices_task` Task Definition:

*   **`description=(...)`**: This task's instructions are built on the premise that the `real_estate_analyst` has already completed its report. Key aspects include:
    *   **Dependency**: It explicitly states, "Use the completed Real Estate Data Analyst report as context... Do not call the CSV tool." This is vital for the hierarchical process.
    *   **Price-Estimation Approach**: Explains how various property attributes (state, city, size, etc.) can be used for comparative pricing.
    *   **Market Segmentation**: Instructs the agent to divide the sample into logical market groups (e.g., lower-priced, mid-priced, luxury) based on the first agent's distribution analysis.
    *   **Comparable-Listing Method**: Details how to evaluate properties by comparing them to similar listings based on various characteristics.
    *   **Representative Property Evaluations**: Requires the agent to select and evaluate *specific properties identified in the first agent's report*, providing estimated price ranges and influencing factors. This prevents the agent from hallucinating new properties.
    *   **High- and Low-Price Observations**: Discusses unusual listings relative to their size or price per square foot.
    *   **Limitations**: Clearly outlines the inherent limitations of the analysis (sample size, missing features like property age, condition, school districts, etc.).
    *   **Final Output**: Specifies the required structure of the final report (executive summary, methodology, market segments, etc.).
*   **`expected_output=(...)`**: Defines the expected comprehensive report structure.
*   **`agent=property_price_analyst`**: Assigns this task to the `property_price_analyst`.
*   **`context=[analyze_real_estate_task]`**: **This is a crucial element for establishing the hierarchical workflow.** It tells CrewAI that the output of `analyze_real_estate_task` (the first agent's report) should be passed as context to this second task. This allows the `property_price_analyst` to build directly on the findings of the `real_estate_analyst` without re-analyzing the raw data, ensuring a logical flow of information and expertise between agents.

In [ ]:
# Create a CrewAI crew containing both agents and both tasks
real_estate_analysis_crew = Crew(
    agents=[
        real_estate_analyst,
        property_price_analyst
    ],

    tasks=[
        analyze_real_estate_task,
        estimate_property_prices_task
    ],

    # Use the hierarchical workflow required by the assignment
    process=Process.hierarchical,

    # Gemini will also manage the hierarchical workflow
    manager_llm=gemini_llm,

    verbose=True
)

print("Hierarchical real estate crew created successfully.")

This cell orchestrates the entire multi-agent workflow by creating the `CrewAI` crew, which acts as the conductor for the agents and their tasks.

*   **`real_estate_analysis_crew = Crew(...)`**: This line initializes the `Crew` object.
    *   **`agents=[real_estate_analyst, property_price_analyst]`**: This list includes all the agents participating in the crew. The `Crew` will manage their collaboration.
    *   **`tasks=[analyze_real_estate_task, estimate_property_prices_task]`**: This list specifies the tasks that the crew needs to complete. The `Crew` will ensure these tasks are executed in the correct order and context.
    *   **`process=Process.hierarchical`**: This is a critical setting. It defines the workflow strategy for the crew. In a `hierarchical` process, tasks are typically executed sequentially, and the output of one task often serves as the input or context for a subsequent task. This contrasts with a `sequential` process where tasks are strictly chained without a manager LLM, or a `concurrent` process where tasks might run in parallel.
    *   **`manager_llm=gemini_llm`**: In a `hierarchical` process, a `manager_llm` is required. This LLM acts as an orchestrator, overseeing the entire workflow, deciding which agent works on which task, and how information flows between them. It uses its intelligence to manage the overall progression, ensuring tasks are completed efficiently and correctly, using the `gemini_llm` configured earlier.
    *   **`verbose=True`**: Enables verbose logging for the entire crew, providing insights into the manager's decisions and the flow of the process.

Once this cell is executed, the `real_estate_analysis_crew` object is ready to be kicked off, initiating the collaborative work of the agents to achieve the project's goals.

In [ ]:
# Start the two-agent real estate analysis
print("## Starting the Real Estate Analysis Crew")

try:
    result = await real_estate_analysis_crew.kickoff_async()

    print(
        "\n## Final Real Estate Analysis "
        "and Property Pricing Report:"
    )

    print(result)

except Exception as error:
    print("\nThe CrewAI workflow did not complete.")
    print(f"Error: {error}")

    print(
        "\nIf the error is 429 RESOURCE_EXHAUSTED, wait at least "
        "one minute before trying again. Do not repeatedly rerun "
        "the cell while the quota is resetting."
    )

This cell initiates the execution of the entire CrewAI workflow and handles its output or potential errors.

1.  **`print("## Starting the Real Estate Analysis Crew")`**: This line simply prints a message to indicate that the crew's work is beginning.
2.  **`try-except` Block**: The core of this cell is a `try-except` block, which is used for robust error handling. This is especially important when interacting with external APIs (like the Gemini LLM) where issues such as quota limits or network errors can occur.
3.  **`result = await real_estate_analysis_crew.kickoff_async()`**: This is the command that starts the CrewAI workflow.
    *   `real_estate_analysis_crew`: This is the `Crew` object created in the previous cell, which encapsulates all agents and tasks.
    *   `kickoff_async()`: This method asynchronously executes the crew's defined process (in this case, `hierarchical`). It will run through all the agents and tasks, managing their interactions and data flow, and eventually return the final output of the last task in the crew.
    *   `await`: Since `kickoff_async()` is an asynchronous function, `await` is used to pause the execution of this cell until the entire crew has completed its work.
4.  **Print Final Report**: If the crew completes successfully, the final report generated by the `estimate_property_prices_task` (the last task in the crew) is stored in the `result` variable and then printed to the console.
5.  **Error Handling**: If any `Exception` occurs during the crew's execution, the `except` block catches it.
    *   It prints an error message, including the specific `error` details.
    *   It provides crucial advice regarding `429 RESOURCE_EXHAUSTED` errors, which commonly occur when API rate limits are exceeded. This advises the user to wait before retrying to allow the quota to reset, preventing repeated failures.